# V4 Corrected-Sequence Retraining and Controlled Comparison

This notebook trains V4 from scratch on `corrected_adaptive_sequences_v2` while keeping the selected V4 architecture and negative strategy fixed. It then evaluates three controlled variants on the same locked A2 queries, candidate catalog, time eligibility, seen-item masks, and exact targets:

1. legacy V4 + legacy sequence;
2. legacy V4 + corrected sequence;
3. corrected V4 + corrected sequence.

Use an **A100 High-RAM** runtime when available; L4 High-RAM is the minimum practical fallback. Checkpoints and reports go to private Hugging Face storage. Drive receives only lightweight reports.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, os, shutil, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT = DRIVE_ROOT / 'data/gold/adaptive/v1'
BASE_GOLD_ROOT = DRIVE_ROOT / 'data/gold/two_tower/v1_colab'
PROTOCOL_DIR = ADAPTIVE_ROOT / 'evaluation_protocol/v2'
WORK_ROOT = Path('/content/recsys_v4_corrected_v1')
SOURCE_ROOT = WORK_ROOT / 'runtime'
HF_SNAPSHOT_ROOT = WORK_ROOT / 'hf_snapshot'
CORRECTED_ROOT = HF_SNAPSHOT_ROOT / 'corrected_sequences/v2'
TRAIN_WORK = WORK_ROOT / 'prepared'
SMOKE_OUTPUT = WORK_ROOT / 'smoke_output'
FULL_OUTPUT = WORK_ROOT / 'artifacts/v4_corrected_v1'
LEGACY_V4_DIR = WORK_ROOT / 'artifacts/legacy_v4'
COMPARE_OUTPUT = WORK_ROOT / 'reports/corrected_v4_comparison'
DRIVE_REPORT_DIR = DRIVE_ROOT / 'artifacts/adaptive/v4_corrected_v1'
for path in (SOURCE_ROOT, TRAIN_WORK, FULL_OUTPUT, LEGACY_V4_DIR, COMPARE_OUTPUT, DRIVE_REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)
print('work root:', WORK_ROOT)

In [ ]:
required = {
    'torch': 'torch', 'pandas': 'pandas', 'pyarrow': 'pyarrow',
    'duckdb': 'duckdb==1.1.3', 'huggingface_hub': 'huggingface_hub>=0.32',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import torch
assert torch.cuda.is_available(), 'Select an A100 High-RAM or L4 High-RAM GPU runtime.'
free_gib = shutil.disk_usage('/content').free / 1024**3
assert free_gib >= 25, f'At least 25 GiB local disk is required; only {free_gib:.1f} GiB is free.'
print('GPU:', torch.cuda.get_device_name(0))
print(f'local disk free: {free_gib:.1f} GiB')

In [ ]:
required_paths = [
    ADAPTIVE_ROOT / 'sequence_examples/train',
    ADAPTIVE_ROOT / 'sequence_examples/validation',
    BASE_GOLD_ROOT / 'transforms/numeric_stats.json',
    BASE_GOLD_ROOT / 'vocabularies/user_active_degree',
    BASE_GOLD_ROOT / 'vocabularies/video_type',
    BASE_GOLD_ROOT / 'vocabularies/upload_type',
    PROTOCOL_DIR / 'queries.parquet',
    PROTOCOL_DIR / 'manifest.json',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, 'Missing Drive inputs:\n' + '\n'.join(missing_paths)
protocol_manifest = json.loads((PROTOCOL_DIR / 'manifest.json').read_text())
assert protocol_manifest['builder_revision'] == '2.1-session-history'
print('Drive/A2 preflight: PASS')
print('A2 queries/users:', protocol_manifest['queries'], protocol_manifest['users'])

## Download verified runtime, corrected sequence delta, and legacy V4

Add `HF_TOKEN` (read) and `HF_WRITE_TOKEN` (write access to `khoa1505/recsys-artifacts`) in Colab Secrets and enable notebook access. The tiny write probe fails before expensive training if permissions are wrong.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from huggingface_hub.utils import EntryNotFoundError

HF_REPO_ID = 'khoa1505/recsys-artifacts'
RUNTIME_FILE = 'runtime/corrected_v4_runtime_v1.zip'
RUNTIME_SHA256 = '0d699a61232f614965c97f1b01b5f02255b9b1d7b9d9b85c80969c18c7de7f62'
LEGACY_V4_FILE = 'v4/v4_inference_fp16.pt'
LEGACY_V4_SHA256 = '4ad49b3e6f9f8b5d45a6c92f31d8d9b27f558e1746613efb44a23db3cd456a26'

def normalize_token(raw):
    if isinstance(raw, bytes): raw = raw.decode()
    if isinstance(raw, str):
        raw = raw.strip()
        return raw if raw.startswith('hf_') else None
    if isinstance(raw, dict):
        for value in raw.values():
            token = normalize_token(value)
            if token: return token
    return None

def secret(name):
    try: return normalize_token(userdata.get(name))
    except Exception: return None

HF_TOKEN = secret('HF_TOKEN') or normalize_token(getpass('Paste Hugging Face READ token: '))
HF_WRITE_TOKEN = secret('HF_WRITE_TOKEN') or normalize_token(getpass('Paste Hugging Face WRITE token: '))
assert HF_TOKEN and HF_WRITE_TOKEN
api = HfApi(token=HF_WRITE_TOKEN)
probe = WORK_ROOT / 'write_probe.txt'; probe.write_text('v4-corrected-write-probe\n')
api.upload_file(repo_id=HF_REPO_ID, repo_type='model', path_or_fileobj=str(probe), path_in_repo='v4_corrected/.write_probe.txt', commit_message='Verify corrected V4 write access')
api.delete_file(repo_id=HF_REPO_ID, repo_type='model', path_in_repo='v4_corrected/.write_probe.txt', commit_message='Remove corrected V4 write probe')
print('Hugging Face write preflight: PASS')

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        while block := stream.read(8 * 1024 * 1024): digest.update(block)
    return digest.hexdigest()

runtime_zip = Path(hf_hub_download(HF_REPO_ID, RUNTIME_FILE, repo_type='model', token=HF_TOKEN, force_download=True))
legacy_v4 = Path(hf_hub_download(HF_REPO_ID, LEGACY_V4_FILE, repo_type='model', token=HF_TOKEN))
assert sha256(runtime_zip) == RUNTIME_SHA256
assert sha256(legacy_v4) == LEGACY_V4_SHA256
snapshot_download(HF_REPO_ID, repo_type='model', token=HF_TOKEN, local_dir=HF_SNAPSHOT_ROOT, allow_patterns=['corrected_sequences/v2/**'])
corrected_manifest = json.loads((CORRECTED_ROOT / 'manifest.json').read_text())
assert corrected_manifest['quality']['passed'] is True
assert corrected_manifest['quality']['rows'] == 9987780
legacy_target = LEGACY_V4_DIR / 'best_model.pt'
legacy_target.unlink(missing_ok=True); legacy_target.symlink_to(legacy_v4.resolve())
print('runtime/data/legacy V4 integrity: PASS')

In [ ]:
if SOURCE_ROOT.exists(): shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    assert archive.testzip() is None
    archive.extractall(SOURCE_ROOT)
TRAIN_SCRIPT = SOURCE_ROOT / 'scripts/train_corrected_adaptive_v4.py'
COMPARE_SCRIPT = SOURCE_ROOT / 'scripts/compare_corrected_adaptive_v4.py'
assert TRAIN_SCRIPT.exists() and COMPARE_SCRIPT.exists()
compile(TRAIN_SCRIPT.read_text(), str(TRAIN_SCRIPT), 'exec')
compile(COMPARE_SCRIPT.read_text(), str(COMPARE_SCRIPT), 'exec')
print('runtime extraction: PASS')

## Smoke train
This validates the real Drive/Hugging Face joins, tensor encoding, forward/backward pass, and checkpoint save before the full run.

In [ ]:
shutil.rmtree(SMOKE_OUTPUT, ignore_errors=True)
smoke_work = WORK_ROOT / 'smoke_prepared'; shutil.rmtree(smoke_work, ignore_errors=True)
smoke_command = [
    sys.executable, str(TRAIN_SCRIPT),
    '--adaptive-root', str(ADAPTIVE_ROOT), '--base-gold-root', str(BASE_GOLD_ROOT),
    '--corrected-root', str(CORRECTED_ROOT), '--work-dir', str(smoke_work), '--output-dir', str(SMOKE_OUTPUT),
    '--max-train-examples', '2000', '--max-validation-examples', '500', '--max-item-pool-examples', '20000',
    '--batch-size', '128', '--eval-batch-size', '128', '--negative-sample-count', '256',
    '--validation-batches', '2', '--epochs', '1', '--checkpoint-every', '1', '--device', 'cuda',
]
subprocess.run(smoke_command, check=True)
assert (SMOKE_OUTPUT / 'best_model.pt').exists()
print('real-data smoke training: PASS')
shutil.rmtree(smoke_work, ignore_errors=True); shutil.rmtree(SMOKE_OUTPUT, ignore_errors=True)

## Full corrected V4 training

The script uploads a resumable FP16 checkpoint every two epochs and final best/latest checkpoints. If Colab dies, rerun this notebook: it automatically downloads those checkpoints and resumes weights/history. Optimizer state is intentionally not persisted because it would add several GB; after resume, AdamW restarts.

In [ ]:
def optional_download(filename):
    try:
        return Path(hf_hub_download(HF_REPO_ID, filename, repo_type='model', token=HF_TOKEN, force_download=True))
    except EntryNotFoundError:
        return None

local_latest = FULL_OUTPUT / 'latest_resume_fp16.pt'
local_best = FULL_OUTPUT / 'best_model.pt'
resume_checkpoint = local_latest if local_latest.exists() else optional_download('v4_corrected/v1/latest_resume_fp16.pt')
prior_best = local_best if local_best.exists() else optional_download('v4_corrected/v1/best_model.pt')
print('resume:', resume_checkpoint)
print('prior best:', prior_best)
command = [
    sys.executable, str(TRAIN_SCRIPT),
    '--adaptive-root', str(ADAPTIVE_ROOT), '--base-gold-root', str(BASE_GOLD_ROOT),
    '--corrected-root', str(CORRECTED_ROOT), '--work-dir', str(TRAIN_WORK), '--output-dir', str(FULL_OUTPUT),
    '--batch-size', '1024', '--eval-batch-size', '1024', '--epochs', '16',
    '--learning-rate', '0.0006', '--weight-decay', '0.00001',
    '--negative-sample-count', '8192', '--popular-negative-fraction', '0.7',
    '--validation-batches', '50', '--early-stopping-patience', '4', '--checkpoint-every', '2',
    '--hf-repo-id', HF_REPO_ID, '--hf-path-prefix', 'v4_corrected/v1', '--device', 'cuda',
]
if resume_checkpoint: command += ['--resume-checkpoint', str(resume_checkpoint)]
if prior_best: command += ['--prior-best-checkpoint', str(prior_best)]
env = os.environ.copy(); env['HF_WRITE_TOKEN'] = HF_WRITE_TOKEN
train_log = Path('/content/v4_corrected_training.log')
process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with train_log.open('w') as log:
    for line in process.stdout:
        print(line, end=''); log.write(line); log.flush()
code = process.wait()
if code != 0:
    raise RuntimeError('Corrected V4 training failed:\n' + '\n'.join(train_log.read_text(errors='replace').splitlines()[-200:]))
assert (FULL_OUTPUT / 'best_model.pt').exists()
print('full corrected V4 training: PASS')

In [ ]:
# Persist lightweight training reports; checkpoints are already on Hugging Face.
report_names = ['config.json', 'metrics.json', 'training_history.csv']
for name in report_names:
    source = FULL_OUTPUT / name
    assert source.exists(), source
    shutil.copy2(source, DRIVE_REPORT_DIR / name)
api.upload_folder(
    repo_id=HF_REPO_ID, repo_type='model', folder_path=str(FULL_OUTPUT),
    path_in_repo='v4_corrected/v1/reports', allow_patterns=report_names,
    commit_message='Upload corrected V4 training reports',
)
print('training reports persisted:', DRIVE_REPORT_DIR)

## Controlled A2 comparison
This is the acceptance test. All variants share exact targets, candidates, eligibility, seen masks, and popularity weight.

In [ ]:
shutil.rmtree(COMPARE_OUTPUT, ignore_errors=True); COMPARE_OUTPUT.mkdir(parents=True)
compare_command = [
    sys.executable, str(COMPARE_SCRIPT),
    '--adaptive-root', str(ADAPTIVE_ROOT), '--base-gold-root', str(BASE_GOLD_ROOT),
    '--corrected-root', str(CORRECTED_ROOT), '--protocol-dir', str(PROTOCOL_DIR),
    '--legacy-v4-dir', str(LEGACY_V4_DIR), '--corrected-v4-dir', str(FULL_OUTPUT),
    '--output-dir', str(COMPARE_OUTPUT), '--top-ks', '10,50,100,500,1000',
    '--popularity-blend-weight', '0.02', '--item-batch-size', '8192',
    '--query-batch-size', '16', '--bootstrap-samples', '1000', '--device', 'cuda',
]
compare_log = Path('/content/v4_corrected_comparison.log')
process = subprocess.Popen(compare_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with compare_log.open('w') as log:
    for line in process.stdout:
        print(line, end=''); log.write(line); log.flush()
code = process.wait()
if code != 0:
    raise RuntimeError('Corrected comparison failed:\n' + '\n'.join(compare_log.read_text(errors='replace').splitlines()[-240:]))
comparison = json.loads((COMPARE_OUTPUT / 'corrected_v4_comparison.json').read_text())
print(json.dumps(comparison['paired_deltas'], indent=2))

In [ ]:
# Persist and download only compact reports. Large per-query Parquet remains optional on Hugging Face.
small_report_dir = WORK_ROOT / 'small_reports'; shutil.rmtree(small_report_dir, ignore_errors=True); small_report_dir.mkdir()
for name in ('corrected_v4_comparison.json', 'summary.csv', 'segments.csv'):
    shutil.copy2(COMPARE_OUTPUT / name, small_report_dir / name)
shutil.copy2(train_log, small_report_dir / train_log.name)
shutil.copy2(compare_log, small_report_dir / compare_log.name)
api.upload_folder(repo_id=HF_REPO_ID, repo_type='model', folder_path=str(small_report_dir), path_in_repo='v4_corrected/v1/evaluation', commit_message='Upload corrected V4 controlled evaluation')
for path in small_report_dir.iterdir(): shutil.copy2(path, DRIVE_REPORT_DIR / path.name)
archive = Path(shutil.make_archive('/content/v4_corrected_v1_report', 'zip', small_report_dir))
print('report archive:', archive, f'{archive.stat().st_size / 1024**2:.2f} MiB')
files.download(str(archive))
delta = comparison['paired_deltas']['corrected_retraining_delta']['hitrate@1000']
accepted = delta['ci95_low'] > 0
print('corrected V4 accepted over legacy V4 on corrected inputs:', accepted)
print('HR@1000 delta:', delta)